## 네이버 카페 내용 크롤링

네이버 가벽비교 사이트 [가격비교](https://search.shopping.naver.com/search) 에서 원하는 검색어를 통해 가격비교 접속하고 리뷰를 찾습니다.



In [19]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from time import sleep
import random
import pandas as pd

driver = webdriver.Chrome()
url = 'https://section.cafe.naver.com/ca-fe/home'
driver.get(url)

### 콘텐츠 가져오기

먼저 카페 요약 게시물 에서 내용를 가져오기위한 상위 요소를 찾아야 합니다.

<rigth><img src="https://drive.google.com/thumbnail?id=1MpRio3YTWNwpj7T20K9VsprTQxXd-oPl&sz=w1000" width="600"/></rigth>


#### 원하는 탭으로 제어 변환

드라이버에서 원하는 페이지를 접속하다 보면 새로운 탭으로 사이트가 열리는 경우가 생깁니다. 기본적으로 `driver`는 기존 탭에 머물러 있기때문에 새로 열린 탭으로 제어를 바꾸어 주어야 합니다.

In [21]:
# 드라이버에 있는 탭 목록 반환
all_handles = driver.window_handles

# -번째 탭으로 전환
driver.switch_to.window(all_handles[0])

#### 상위 요소 목록 찾기

페이지의 css 소스는 항상 달라지므로 상시 확인이 필요합니다.

In [22]:
# CSS_SELECTOR
cs = 'div.ArticleItem'
reviews = driver.find_elements(By.CSS_SELECTOR, cs)
print(len(reviews))

12


> CSS_SELECTOR를 띄어쓰기로 연속하여 입력하면 순서대로 상위요소 부터 접근

In [23]:
# XPATH
xp = '//*[@id="mainContainer"]/div[1]/div[2]/div[2]/div/div'
reviews = driver.find_elements(By.XPATH, xp)
print(len(reviews))


12


#### 하위 요소 내용 가져오기

In [43]:
def get_content(xp=None, css=None):
    # 입력된 경로 형태에 맞게 요소를 찾음
    if xp is not None:
        reviews = driver.find_elements(By.XPATH, xp)
    elif css is not None:
        reviews = driver.find_elements(By.CSS_SELECTOR, css)
    else:
        return

    # 각 리뷰에서 하위 요소를 가져옴
    craw = []
    for rv in reviews:
        content = {}
        info = rv.find_elements(By.CSS_SELECTOR, 'div.reviewItemProfile_info_data__l3ebj span')
        content['title'] = rv.find_element(By.CSS_SELECTOR,
                                          'strong.title').get_attribute('innerText')
        content['text'] = rv.find_element(By.CSS_SELECTOR,
                                          'p.text').get_attribute('innerText')
        content['rat'] = rv.find_element(By.CSS_SELECTOR,
                                         'span.cafe_name').get_attribute('innerText')
        content['date'] = rv.find_element(By.CSS_SELECTOR,
                                         'span.date').get_attribute('innerText')

        # 썸네일을 없는 경우 None으로 대체하기 위해 예외 처리
        try:
            content['img'] = rv.find_element(By.CSS_SELECTOR,
                                             'a.thumbnail source').get_attribute('srcset')
        except:
            content['img'] = None
            
        craw.append(content)
    return craw

get_content(css=cs)

[{'title': '제주도 맛집정보좀 부탁드려요~',
  'text': '하나씩 추천해주세요~ 예전에는 산방식당은 제주도 가면 꼭 갔었는데~ 요즘은 제주 맛집이라고 해서 찾아보면.. 종류가 너무 다양해져있는 것 같아요~ 회원님중에서 연돈 드셔보신분 계시나요?',
  'rat': '칼사랑조리사모임 ◇일식,양식,한식,중식◇ No.1:요리사커뮤니티',
  'date': '2022.06.29.',
  'img': None},
 {'title': '최홍만 제주도 맛집 검증후기',
  'text': '최홍만 제주도 맛집이라는 검색어로 찾다가 문개어멍의 방송 정보를... 않아도 음식 자체로 다시 찾을 이유가 충분했던 문개어멍이었어요. ▶ 업체명 : 문개어멍 ▶ 업체 주소 : 제주 제주시 애월읍 가문동길 38...',
  'rat': '맛슐랭코리아',
  'date': '2026.08.10.',
  'img': 'https://cafeptthumb-phinf.pstatic.net/MjAyNjA4MTBfMjI4/MDAxNzg2MzM5NDI0MTQ1.UGVFCdCj5FJLUOzO7JebdE1vcaYnqboYXJOwBydGoNwg.R3tp1TcOB50bpsoFL4eXVYCGZIo3n8ruYIpCs9plcgIg.JPEG/%EC%82%AC%EC%A7%845_%EB%8C%80%ED%91%9C.jpg?type=ff200_200'},
 {'title': '제주도 맛집 여기저기 추천이요!!!',
  'text': '이번에 제주도가서 먹은 맛집들 최신업뎃!! 해요!! 1.연돈 골목식당 때문에 아직도 사람이 많더라구요ㅜㅜ 8시 좀 넘어서 줄서서 예약하고 먹었어요 등심인데도 씹기도전에 고기가 치아에 닿자마자 잘려요ㅠㅠㅠㅠ...',
  'rat': '다이렉트 결혼준비',
  'date': '2020.10.14.',
  'img': 'https://cafeptthumb-phinf.pstatic.net/MjAyMDEwMTRfNzMg/MDAxNjAyNjM5OTk4Mzgz.bNTPxvLC31-CDKjW

> find_elements 결과에서 순서를 활용하여 요소를 찾는 방법도 유용

### 페이지 변경하여 내용 노출
- url 링크에 페이지 번호가 노출되면 문자열 포맷으로 `get()` 함수를 반복 호출 가능 (쉬움)
- url 링크에 페이지 번호가 없으면 직접 버튼을 누르게 소스코드를 구현 (어려움)

<rigth><img src="https://drive.google.com/thumbnail?id=1-_rxsmlczs6aE_zwqjFvQeRFTOqUcoE9&sz=w1000" width="500"/></rigth>


#### 번호 목록 요소의 구성 파악

In [34]:
gap_xp = '//*[@id="mainContainer"]/div[1]/div[2]/div[3]/button'
pag = driver.find_elements(By.XPATH, gap_xp)
print(len(pag))
pag[0].get_attribute('innerText')

11


'1'

> 처음 페이지 번호 요소를 가져온 경우 11개의 번호를 가져오며 마지막 번호의 요소는 다음페이지 버튼

In [35]:
pag[10].click()
sleep(1)
pag = driver.find_elements(By.XPATH, gap_xp)
print(len(pag))
pag[0].get_attribute('innerText')

12


''

> '다음' 버튼을 클릭한 다음 바뀐 번호 목록을 가져오면 12개의 목록을 가져옴(이전 버튼이 추가됨)   

#### 반복문을 통해 번호 클릭하여 수집

In [44]:

i = 1
result = []
while True:
    pag = driver.find_element(By.XPATH, f'{gap_xp}[{i}]')
    pt = pag.get_attribute('innerText')
    print(pt)
    pag.click()
    if pt == '':
        i = 2
    else:
        i += 1
    sleep(1)
    result.extend(get_content(css=cs))
    if len(result) > 1000:
        break


1
2
3
4
5
6
7
8
9
10

11
12
13
14
15
16
17
18
19
20

21
22
23
24
25
26


KeyboardInterrupt: 

> 클릭한 버튼의 내용을 가져와 검사하여 다음 클릭할 순서를 선정하는 방식 (내용으로 구분가능한 경우)     
> 해당 방법 외에도 단순 순서를 고려한 방법과 목록 개수를 고려하여 조건문을 구성하고 클릭하는 방법등이 사용됨   

In [45]:
df = pd.DataFrame(result)
df

,title,text,rat,date,img
0,제주시 맛집 추천,이번엔 제주시 맛집의 몸국을 먹어봤는데 해조류를 넣어 끓인 제주 대표 음식이에요. ...,부산 경남 맘스홀릭 육아 생활정보 체험단 이벤트 중고거래,2026.08.07.,https://cafeptthumb-phinf.pstatic.net/MjAyNjA4...
1,제주도맛집추천해주세요,6월 제주도 비행기 예약했어요 비가안와야할텐데..! 맛집 추천해주세요ㅠㅠㅠ,"맘이베베 (핫딜, 육아, 놀이)",2026.05.26.,None
2,제주에서 맛집하나씩추천부탁드립니다,가족들과 제주왔는데요 바다수영은 그만하고 밥과카페 실내 돌려고합니다1박2일동안요 혹...,[느영나영] 제주도여행 대표카페 제주맛집 렌터카 숙소 여행지도,2026.07.20.,None
3,광고사절)제주도 맛집 진짜 있나요??,리뷰가 넘 좋아서 갔는데 대실망한 적이 한두번이 아니에요. 리뷰적으면 뭐 주는곳은 ...,"맘이베베 (핫딜, 육아, 놀이)",2026.05.28.,None
4,제주도 함덕 맛집 list,"지난 주 다녀온 제주도 함덕 맛집 제주섬고등어쌈밥 고등어쌈밥부터 갈치조림, 갈치국,...",부산 경남 맘스홀릭 육아 생활정보 체험단 이벤트 중고거래,2026.08.06.,https://cafeptthumb-phinf.pstatic.net/MjAyNjA4...
...,...,...,...,...,...
319,제주도 맛집 든든한 아침식사 추천 성산 고기국수 꽃가람,선사한 제주도 고기국수 맛집이라 소개해 볼게요. 꽃가람 제주성산일출봉점 주소: 제주...,[느영나영] 제주도여행 대표카페 제주맛집 렌터카 숙소 여행지도,2025.08.08.,https://cafeptthumb-phinf.pstatic.net/MjAyNTA4...
320,제주도 맛집알려주세요,공항이나 성산 우도쪽만요 갈치회 갈치찜 고등어회 딱새우회 고기국수 요정도$ 정했어요...,[울산] 맛집멋집,2024.03.19.,None
321,제주도 맛집 신상버전 직접가본후기 (초스압주의) 제주시 서귀포시,음식의느낌 저또한 매우 만족한곳 별 9개 ★★★★★★★★★ 제주시 노형동 모이세해장...,●디젤매니아● 대한민국 일등 패션 커뮤니티 디매인 DMAIN,2021.07.02.,https://cafeptthumb-phinf.pstatic.net/MjAyMTA2...
322,[제주도 맛집 추천] 제주시 맛집 화성식당 방문 후기,생각나는 음식이 있습니다.. 바로 접작뼈국인데요! 뭔가 동네에서 먹으면 제주도에서 ...,"요즘웨딩, 결혼준비 이제 요즘답게 :웨딩홀,스드메,결혼준비어플",2026.07.09.,https://cafeptthumb-phinf.pstatic.net/MjAyNjA3...


In [46]:
df.to_csv('./data/craw.csv', index=False)

### 이미지 추가 수집
이미지 태그인 `<img>` 내부의 `src` 요소에 있는 url 값을 저장했다면 해당 주소를 통해 이미지를 다운로드 할 수있습니다.

**urllib 모듈 활용**
urllib 라이브러리를 활용하면 url 주소로 부터 연결된 데이터를 컴퓨터에 다운로드 가능합니다.

- `urllib.request.urlretrieve(url,file)` : `url` 경로에있는 데이터를 `file`에 다운로드

In [47]:
import urllib.request as urq
import os

# 저장 폴더 생성
os.makedirs('./image', exist_ok=True)

i = 0
for link in df['img']:
    try:
        # 이미지 링크(link)를 사용하여 이미지를 다운로드하고, 지정된 경로에 저장
        urq.urlretrieve(link, f'image/img_{i}.jpg')
    except Exception as e:
        print(f'{i}번째 사진 에러: {e}')
    i += 1

1번째 사진 에러: expected string or bytes-like object
2번째 사진 에러: expected string or bytes-like object
3번째 사진 에러: expected string or bytes-like object
6번째 사진 에러: expected string or bytes-like object
8번째 사진 에러: expected string or bytes-like object
9번째 사진 에러: expected string or bytes-like object
12번째 사진 에러: expected string or bytes-like object
16번째 사진 에러: expected string or bytes-like object
18번째 사진 에러: expected string or bytes-like object
20번째 사진 에러: expected string or bytes-like object
23번째 사진 에러: expected string or bytes-like object
30번째 사진 에러: expected string or bytes-like object
31번째 사진 에러: expected string or bytes-like object


KeyboardInterrupt: 

# 실습 해보기


#### [실습1]

네이버 검색창의 [뉴스](https://search.naver.com/search.naver?ssc=tab.news.all&where=news&sm=tab_jum&query=)또는 [블로그](https://search.naver.com/search.naver?ssc=tab.blog.all&sm=tab_jum&query=)에서 크롤링 하기

1. 위의 검색에서 원하는 키워드로 검색하여 나온 요약 내용을 드래그를 통해 100개 이상 크롤링
    * 뉴스 게시물에 내용에는 제목, 요약글, 날짜, 언론사 포함
    * 블로그 게시물에 내용에는 제목, 요약글, 날짜, 블로그작성자 포함


#### [실습2]

[네이버블로그](https://section.blog.naver.com/BlogHome.naver?directoryNo=0&currentPage=1&groupId=0)에서 크롤링 하기

1. 위의 블로그 홈에서 원하는 키워드로 검색하여 나온 게시물 내용을 10페이지 이상 크롤링
    * 게시물에 내용에는 제목, 요약글, 날짜, 작성자, 이미지 링크(url)가 포함
2. [심화] 이미지 링크를 활용하여 첫번째 썸네일 이미지를 저장
3. [심화] 날짜 검색을 활용하여 기간을 월별로 나누어 최근 6개월의 글을 크롤링

#### [참고] 날짜 만들기

In [ ]:
dateforms = []

daterng = pd.date_range(start='1/1/2020', end = '6/1/2022', freq='MS').strftime('%Y%m%d')

for i in range(len(daterng) - 1):
    dateforms.append(f'{daterng[i]}to{daterng[i+1]}')
dateforms

['20200101to20200201',
 '20200201to20200301',
 '20200301to20200401',
 '20200401to20200501',
 '20200501to20200601',
 '20200601to20200701',
 '20200701to20200801',
 '20200801to20200901',
 '20200901to20201001',
 '20201001to20201101',
 '20201101to20201201',
 '20201201to20210101',
 '20210101to20210201',
 '20210201to20210301',
 '20210301to20210401',
 '20210401to20210501',
 '20210501to20210601',
 '20210601to20210701',
 '20210701to20210801',
 '20210801to20210901',
 '20210901to20211001',
 '20211001to20211101',
 '20211101to20211201',
 '20211201to20220101',
 '20220101to20220201',
 '20220201to20220301',
 '20220301to20220401',
 '20220401to20220501',
 '20220501to20220601']

In [ ]:
# 현재 열려 있는 모든 탭(창)의 핸들을 가져옵니다.
all_handles = driver.window_handles

# 두 번째 탭으로 전환합니다. (인덱스는 0부터 시작하므로 두 번째 탭은 인덱스 1입니다.)
driver.switch_to.window(all_handles[1])

# 두 번째 탭에서 요소를 찾습니다. (예시: 페이지 제목 출력)
print(driver.title)

# 이제 두 번째 탭에서 find_element 또는 find_elements를 사용하여 원하는 요소를 찾을 수 있습니다.
# element = driver.find_element(By.CSS_SELECTOR, '원하는 CSS 셀렉터')
# elements = driver.find_elements(By.XPATH, '원하는 XPATH')

# 작업을 마친 후 원래 탭으로 돌아가려면 다음 코드를 사용합니다.
# driver.switch_to.window(all_handles[0])